# ⭐ Píldora 4 · MLflow: Los Archivos Jedi

## Misión
La Orden Jedi necesita conservar el conocimiento generado durante sus misiones de Machine Learning.

En esta práctica trabajaremos con **Digits**, un dataset de imágenes de números escritos a mano. El modelo deberá reconocer qué dígito aparece, del 0 al 9.

Aprenderemos a:

- Crear un **experimento** en MLflow.
- Abrir y nombrar varias **runs**.
- Registrar **parámetros** y **métricas**.
- Guardar **artefactos**, como una matriz de confusión y un informe.
- Guardar el **modelo entrenado**.
- Comparar varias misiones y justificar cuál debería sobrevivir a la Orden 66.

> **Regla de los Archivos Jedi:** si una misión no está registrada, es como si nunca hubiera existido.

---

**Ruta de la píldora:** demo guiada → reto `03_Orden_66_RETO.ipynb` → corrección con `02_Archivos_Jedi_SOLUCION.ipynb`.
**Entorno validado:** Python 3.12, MLflow 3.14.0, scikit-learn 1.9.0, pandas 2.3.3 y matplotlib 3.11.0.


## Correspondencia narrativa

| Archivos Jedi | MLflow |
|---|---|
| Archivo de una campaña | Experimento |
| Una misión concreta | Run |
| Estrategia elegida | Parámetros |
| Resultado de la misión | Métricas |
| Informes, mapas y hologramas | Artefactos |
| Jedi entrenado que puede recuperarse | Modelo guardado |

La narrativa es solo una ayuda para recordar los conceptos. En el código utilizaremos los nombres técnicos reales.


## 1. Instalación

In [ ]:
# Entorno validado el 15-07-2026. Ejecuta esta celda una sola vez en Colab.
%pip install -q "mlflow==3.14.0" "scikit-learn==1.9.0" "pandas==2.3.3" "matplotlib==3.11.0"

## 2. Importaciones

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow.models import infer_signature

from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

print("MLflow:", mlflow.__version__)


## 3. Configurar MLflow

`set_tracking_uri` indica dónde se almacenarán los datos de tracking.  
`set_experiment` selecciona o crea el experimento que agrupará nuestras runs.


In [ ]:
# SQLite conserva metadatos y modelos de forma consistente en Colab y local.
tracking_db = Path("mlflow_jedi_digits.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")

experiment_name = "Archivos_Jedi_Orden_66_Digits"
experiment = mlflow.set_experiment(experiment_name)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", experiment_name)
print("Experiment ID:", experiment.experiment_id)

## 4. Cargar y dividir los datos

Usaremos el dataset **Digits** de Scikit-learn. Contiene 1.797 imágenes de números escritos a mano de 8 × 8 píxeles. Cada imagen se representa mediante 64 valores y pertenece a una de las diez clases posibles: del 0 al 9.

In [ ]:
data = load_digits(as_frame=True)

X = data.data
y = data.target

test_size = 0.20
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state,
    stratify=y,
)

print("Dataset: Digits")
print("Muestras totales:", len(X))
print("Entrenamiento:", X_train.shape)
print("Test:", X_test.shape)
print("Clases:", list(data.target_names))

# Mostramos una imagen para entender qué está clasificando el modelo.
plt.imshow(data.images[0], cmap="gray")
plt.title(f"Dígito real: {data.target[0]}")
plt.axis("off")
plt.show()

## 5. Definir las misiones

Cada diccionario contiene una configuración distinta. `n_estimators` y `max_depth` son hiperparámetros porque los decidimos antes del entrenamiento.

Las configuraciones se han ajustado para que la comparación sea razonable:

- Tatooine es más ligera.
- Coruscant busca un equilibrio.
- Mustafar utiliza más capacidad y recursos.

In [ ]:
missions = [
    {
        "run_name": "Mision_Tatooine",
        "n_estimators": 50,
        "max_depth": 8,
    },
    {
        "run_name": "Mision_Coruscant",
        "n_estimators": 100,
        "max_depth": 10,
    },
    {
        "run_name": "Mision_Mustafar",
        "n_estimators": 300,
        "max_depth": None,
    },
]

pd.DataFrame(missions)

## 6. Función completa de tracking

In [ ]:
def execute_mission(config):
    """Entrena una misión y registra toda su evidencia en MLflow."""

    run_name = config["run_name"]
    n_estimators = config["n_estimators"]
    max_depth = config["max_depth"]

    # Cada misión tendrá su propia carpeta temporal de artefactos.
    artifact_dir = Path("artifacts_jedi") / run_name
    artifact_dir.mkdir(parents=True, exist_ok=True)

    with mlflow.start_run(run_name=run_name) as run:
        # Etiquetas descriptivas: organizan, pero no afectan al modelo.
        mlflow.set_tags(
            {
                "universo": "Star Wars",
                "operacion": "Orden 66",
                "tipo_modelo": "RandomForestClassifier",
                "dataset": "Digits",
            }
        )

        # 1. Crear y entrenar el modelo.
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state,
        )
        model.fit(X_train, y_train)

        # 2. Realizar predicciones.
        y_pred = model.predict(X_test)

        # 3. Calcular métricas globales.
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        )

        # 4. Registrar parámetros: la estrategia elegida antes de entrenar.
        mlflow.log_params(
            {
                "n_estimators": n_estimators,
                "max_depth": str(max_depth),
                "test_size": test_size,
                "random_state": random_state,
                "dataset": "digits",
            }
        )

        # 5. Registrar métricas: resultados obtenidos después de entrenar.
        mlflow.log_metrics(
            {
                "accuracy": accuracy,
                "precision_weighted": precision,
                "recall_weighted": recall,
                "f1_weighted": f1,
            }
        )

        # 6. Crear y guardar una matriz de confusión.
        confusion_path = artifact_dir / "matriz_confusion.png"

        ConfusionMatrixDisplay.from_predictions(
            y_test,
            y_pred,
            display_labels=data.target_names,
            cmap="Blues",
        )
        plt.title(f"Matriz de confusión · {run_name}")
        plt.tight_layout()
        plt.savefig(confusion_path, dpi=150)
        plt.close()

        mlflow.log_artifact(str(confusion_path), artifact_path="evidencias")

        # 7. Crear un informe por dígito como segundo artefacto.
        report = classification_report(
            y_test,
            y_pred,
            target_names=[str(label) for label in data.target_names],
            zero_division=0,
            output_dict=True,
        )
        report_path = artifact_dir / "classification_report.json"

        with open(report_path, "w", encoding="utf-8") as file:
            json.dump(report, file, indent=2, ensure_ascii=False)

        mlflow.log_artifact(str(report_path), artifact_path="evidencias")

        # 8. Guardar el modelo entrenado como Logged Model (MLflow 3).
        signature = infer_signature(X_train, model.predict(X_train))
        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name="modelo_jedi",
            signature=signature,
            input_example=X_train.head(3),
        )

        return {
            "run_id": run.info.run_id,
            "run_name": run_name,
            "accuracy": accuracy,
            "precision_weighted": precision,
            "recall_weighted": recall,
            "f1_weighted": f1,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
        }

### Ejecutar las tres misiones

In [ ]:
results = []

for mission in missions:
    result = execute_mission(mission)
    results.append(result)
    print(
        f"{result['run_name']}: "
        f"accuracy={result['accuracy']:.3f}, "
        f"f1={result['f1_weighted']:.3f}"
    )

results_df = pd.DataFrame(results)
results_df


### Comparar las runs desde Python

In [ ]:
experiment = mlflow.get_experiment_by_name(experiment_name)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)

columns = [
    "tags.mlflow.runName",
    "params.n_estimators",
    "params.max_depth",
    "metrics.accuracy",
    "metrics.precision_weighted",
    "metrics.recall_weighted",
    "metrics.f1_weighted",
]

runs_df[columns]


# Si se reejecuta el notebook, conservamos la run más reciente de cada misión.
runs_df = runs_df.drop_duplicates(subset=["tags.mlflow.runName"], keep="first")
runs_df = runs_df.sort_values("metrics.f1_weighted", ascending=False)


### Elegir una misión con un criterio explícito

In [ ]:
# Ejemplo de criterio:
# 1. Priorizar F1.
# 2. Si dos runs empatan o son muy parecidas, preferir la menos compleja.

ranking = results_df.sort_values(
    by=["f1_weighted", "n_estimators"],
    ascending=[False, True],
)

winner = ranking.iloc[0]

print("Misión seleccionada:", winner["run_name"])
print("F1:", round(winner["f1_weighted"], 3))
print("Accuracy:", round(winner["accuracy"], 3))
print("Número de árboles:", winner["n_estimators"])
print("Profundidad máxima:", winner["max_depth"])


## 10. Consultar los Archivos Jedi

La comprobación obligatoria se hace dentro del notebook con `mlflow.search_runs()`, por lo que funciona en Colab sin abrir puertos.

La interfaz web es un extra para Jupyter local. Desde la carpeta del notebook:

```bash
mlflow server --backend-store-uri sqlite:///mlflow_jedi_digits.db --port 5000
```

Después abre `http://127.0.0.1:5000`. En Google Colab esa dirección apunta al servidor remoto y no se abre directamente en tu navegador; durante la clase utiliza la tabla de comparación y los artefactos registrados.


# 🚨 Orden 66: reflexión final

Para que una misión sobreviva debe responder:

- ¿Tiene un nombre reconocible?
- ¿Sabemos con qué parámetros se entrenó?
- ¿Conservamos las métricas?
- ¿Podemos revisar evidencias visuales?
- ¿Está guardado el modelo?
- ¿Podríamos justificar por qué lo elegimos?

> MLflow no decide cuál es el mejor modelo. Nos proporciona la evidencia necesaria para tomar y defender esa decisión.

## Cómo interpretar la decisión

Con `random_state=42`, Mustafar suele obtener el mejor rendimiento global, pero eso no obliga a todos los equipos a elegirla. Una diferencia pequeña puede no compensar el aumento de complejidad.

La respuesta correcta dependerá del holocrón:

- **Rendimiento global:** suele favorecer a Mustafar.
- **Recursos o producción:** puede favorecer a Coruscant o Tatooine si se justifica la pequeña pérdida de rendimiento.
- **Auditoría:** exige comprobar parámetros, métricas, modelo y artefactos.
- **Errores ocultos:** obliga a revisar la matriz y el informe por clase.
- **Rescate prioritario:** obliga a revisar especialmente el recall del dígito 1.

La clave no es acertar un nombre, sino defender la elección con evidencia registrada en MLflow.